In [ ]:
import os
import sys
import json
import csv
import datetime
import numpy as np
import pandas as pd
from tqdm import tqdm

In [ ]:
import geopandas as gpd
import h3
import h3pandas

from shapely import wkt
from shapely.geometry import shape, Polygon
from shapely.geometry.polygon import orient
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

In [ ]:
# Get the parent directory of the current directory (which is 'notebooks')
# and add it to the system path
project_root = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if project_root not in sys.path:
    sys.path.append(project_root)

# Now you can use absolute imports from the project root
from utils.geometry import get_bearing, get_bearing_label, convert_geometry

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Geo Admin boundaries

In [ ]:
THRESHOLD_OVERLAP = 0.5

In [ ]:
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data")

In [ ]:
data_fn = os.path.join(DATA_DIR, 'census\\tl_2024_us_aiannh\\tl_2024_us_aiannh.shp')

In [ ]:
aiannh_gdf = gpd.read_file(data_fn)

In [ ]:
aiannh_gdf.plot(figsize=(13,8))

#### Create a 10km buffer for safety

In [ ]:
aiannh_gdf = aiannh_gdf.to_crs(3857)

In [ ]:
# aiannh_gdf['geom_buff'] = aiannh_gdf.buffer(10000)  # outer bounds 10km
aiannh_gdf['geom_buff'] = aiannh_gdf.buffer(5000)  # inner bounds  5km

In [ ]:
aiannh_gdf['geometry'] = aiannh_gdf['geom_buff']

In [ ]:
del aiannh_gdf['geom_buff']

In [ ]:
# buffered_gdf = gpd.GeoDataFrame(geometry=buffered_mil, crs='EPSG:3857')

In [ ]:
aiannh_gdf = aiannh_gdf.to_crs(4326)

In [ ]:
# buffered_gdf.plot(figsize=(13, 8))

In [ ]:
aiannh_gdf

### H3

In [ ]:
native_hex_ids_r11 = []
native_hex_ids_r5 = []

In [ ]:
try:
    tmp_hex_r5_df = aiannh_gdf.h3.polyfill(5, explode=True)
    native_hex_ids_r5 +=  tmp_hex_r5_df.h3_polyfill.drop_duplicates().to_list()
    
except Exception as e:
    print(e)
    print("error with r5")

In [ ]:
h5_remove_list = [x for x in list(set(native_hex_ids_r5)) if str(x) != 'nan']

In [ ]:
len(h5_remove_list)

In [ ]:
save_df = pd.DataFrame([{"hex_id": x, "reason": "native_land"} for x in h5_remove_list])

In [ ]:
# save_df.to_csv('remove_hexes_r5_native_land_inner.csv')

In [ ]:
hex_aiannh_df = save_df.copy()

In [ ]:
hex_aiannh_df = hex_aiannh_df.set_index('hex_id')

In [ ]:
hex_aiannh_df = hex_aiannh_df.h3.h3_to_geo_boundary()

In [ ]:
hex_aiannh_df = hex_aiannh_df.h3.cell_area(unit='km^2')

In [ ]:
hex_aiannh_df

In [ ]:
hex_aiannh_df.reset_index(inplace=True)

In [ ]:
hex_aiannh_overlap_gdf = gpd.overlay(hex_aiannh_df,\
                               aiannh_gdf[['NAME', 'GEOID', 'ALAND', 'AWATER', 'INTPTLAT', 'INTPTLON', 'geometry']],\
                               how="intersection",\
                               keep_geom_type=False\
                              )

In [ ]:
hex_aiannh_overlap_gdf

In [ ]:
hex_aiannh_overlap_gdf["overlap_area_m"] = hex_aiannh_overlap_gdf.to_crs(crs="EPSG:6933").geometry.area

In [ ]:
hex_aiannh_overlap_gdf["overlap_area"] = hex_aiannh_overlap_gdf["overlap_area_m"] / 1e6

In [ ]:
hex_aiannh_overlap_gdf["overlap_pct"] = hex_aiannh_overlap_gdf["overlap_area"] / hex_aiannh_overlap_gdf["h3_cell_area"]

In [ ]:
hex_aiannh_80pct_gdf = hex_aiannh_overlap_gdf[(hex_aiannh_overlap_gdf["overlap_pct"] > 0.8)]

In [ ]:
hex_aiannh_80pct_df = hex_aiannh_80pct_gdf.filter(['hex_id', 'reason', 'overlap_pct'])

In [ ]:
hex_aiannh_80pct_df = hex_aiannh_80pct_df.rename(columns={'reason': 'reason_inhibited'})

In [ ]:
hex_aiannh_80pct_df['is_inhibited'] = True

In [ ]:
hex_aiannh_80pct_df = hex_aiannh_80pct_df.drop_duplicates(subset=["hex_id"])

In [ ]:
hex_aiannh_80pct_df = hex_aiannh_80pct_df.reset_index(drop=True)

In [ ]:
hex_aiannh_80pct_df

In [ ]:
save_fn = f"remove_hexes_h5_aiannh_{str(int(100*THRESHOLD_OVERLAP))}pct.csv"
save_sql = f"remove_hexes_h5_aiannh_{str(int(100*THRESHOLD_OVERLAP))}pct.sql"

In [ ]:
print(f"INPUT: {save_fn}\tOUTPUT: {save_sql}")

In [ ]:
hex_aiannh_80pct_df.to_csv(save_fn)

In [ ]:
import csv

# --- Configuration ---
# INPUT_FILE = 'C:/Users/bcros/dev/blcrosbie/GeoDataAnalytics/notebooks/remove_hexes_h5_aiannh_80pct.csv'
# INPUT_FILE = 'C:/Users/bcros/dev/blcrosbie/GeoDataAnalytics/notebooks/remove_hexes_h5_mil_80pct.csv'
# INPUT_FILE = 'C:/Users/bcros/dev/blcrosbie/GeoDataAnalytics/notebooks/remove_hexes_r5_native_land.csv'
INPUT_FILE = f'C:/Users/bcros/dev/blcrosbie/GeoDataAnalytics/notebooks/{save_fn}'
# OUTPUT_FILE = 'bulk_update_native_land.sql'
OUTPUT_FILE = save_sql
# OUTPUT_FILE = 'bulk_update_native_land.sql'
# OUTPUT_FILE = 'remove_hexes_h5_mil_80pct.sql'
# OUTPUT_FILE = 'remove_hexes_h5_aiannh_80pct.sql'
TABLE_NAME = 'public.hexes'
# The column name we are setting the boolean flag on
INHIBITED_COLUMN = 'is_inhibited'
# The column name we are setting the reason on
REASON_COLUMN = 'reason_inhibited'
# The column used for the WHERE clause
KEY_COLUMN = 'hex_id'
# --- End Configuration ---


def generate_sql_updates():
    """Reads CSV data and generates a series of SQL UPDATE statements."""
    sql_statements = []

    try:
        with open(INPUT_FILE, mode='r', newline='', encoding='utf-8') as infile:
            # Use DictReader to read data using column headers
            reader = csv.DictReader(infile)
            
            for row in reader:
                hex_id = row.get(KEY_COLUMN)
                reason = row.get('reason_inhibited')

                if not hex_id or not reason:
                    print(f"Skipping row due to missing data: {row}")
                    continue

                # Ensure values are properly quoted for SQL insertion
                # Text values (reason) must be surrounded by single quotes
                # If reason contains a single quote, it needs to be escaped (doubled up)
                safe_reason = reason.replace("'", "''")

                # The SQL UPDATE statement structure
                sql = f"""
UPDATE {TABLE_NAME}
SET 
    {INHIBITED_COLUMN} = TRUE,
    {REASON_COLUMN} = '{safe_reason}'
WHERE 
    {KEY_COLUMN} = '{hex_id}';
"""
                sql_statements.append(sql.strip())

    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_FILE}' not found. Please create it.")
        return []
    except Exception as e:
        print(f"An error occurred during CSV reading: {e}")
        return []

    # Format the statements into a safe transaction block
    full_sql_output = [
        "-- --- Generated SQL Bulk Update Script ---",
        f"-- Generated from {INPUT_FILE} at {len(sql_statements)} rows.",
        "BEGIN;",
        *sql_statements,
        "COMMIT;",
        "-- --- End of Script ---"
    ]

    return full_sql_output


# --- Main Execution ---
generated_sql = generate_sql_updates()

if generated_sql:
    try:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as outfile:
            outfile.write('\n'.join(generated_sql))
        print(f"\nSuccessfully generated {len(generated_sql) - 4} SQL UPDATE statements.")
        print(f"Output saved to '{OUTPUT_FILE}'.")
        print("\nRun this file against your PostgreSQL database to apply changes.")
    except Exception as e:
        print(f"An error occurred during file writing: {e}")

In [ ]:
# try:
#     tmp_hex_r11_df = aiannh_gdf.h3.polyfill(11, explode=True)
#     military_hex_ids_r11 += tmp_hex_r11_df.h3_polyfill.drop_duplicates().to_list()
    
# except Exception as e:
#     print(e)
#     print("error with r11")

In [ ]:
h11_remove_list = [x for x in list(set(military_hex_ids_r11)) if str(x) != 'nan']

In [ ]:
len(h11_remove_list)

In [ ]:
save_df = pd.DataFrame([{"hex_id": x, "reason": "native_land"} for x in h11_remove_list])
save_df.to_csv('remove_hexes_r11_military.csv')

#### Output

In [ ]:
def hex_polygon(hex_id):
    """
    Return shapely Polygon of an H3 cell in lon/lat (EPSG:4326).

    Works with older h3-py that returns (lat, lng) and does NOT support geo_json.
    """
    boundary = h3.cell_to_boundary(hex_id)  # list of (lat, lng)
    # convert to (lon, lat) for shapely / mercantile
    coords = [(lng, lat) for lat, lng in boundary]
    poly = Polygon(coords)
    # Normalize orientation (CCW) just to be consistent
    return orient(poly, sign=1.0)

In [ ]:
# --- 3. Create a Pandas DataFrame ---
removal_df = pd.DataFrame({'hex_id': h5_remove_list})
print(f"Converting {len(removal_df)} H3 IDs to polygon geometries...")
removal_df['geometry'] = removal_df['hex_id'].apply(hex_polygon)

In [ ]:
# --- 5. Create the GeoDataFrame ---
# Use EPSG:4326 (WGS 84) which is the standard CRS for H3.
gdf = gpd.GeoDataFrame(removal_df, geometry='geometry', crs="EPSG:4326")

In [ ]:
# --- 6. Save the GeoDataFrame to a File ---
output_file = 'hex_native_land_validation_r5_inner.geojson'

# GeoJSON is generally the most compatible and robust format for sharing.
# For a shapefile, you would use: gdf.to_file('h3_coverage_validation.shp')
try:
    gdf.to_file(output_file, driver='GeoJSON')
    print("-" * 50)
    print(f"Successfully created GeoJSON file: {output_file}")
    print(f"File saved with {len(gdf)} features.")
    print("You can now load this file into QGIS to visualize your H3 coverage areas.")
except Exception as e:
    print(f"Error saving file: {e}")

# Display a preview of the GeoDataFrame structure
print("\nGeoDataFrame Preview:")
print(gdf.head())

In [ ]:
# --- 3. Create a Pandas DataFrame ---
removal_11_df = pd.DataFrame({'hex_id': h11_remove_list})
print(f"Converting {len(removal_11_df)} H3 IDs to polygon geometries...")
removal_11_df['geometry'] = removal_11_df['hex_id'].apply(hex_polygon)

In [ ]:
# --- 5. Create the GeoDataFrame ---
# Use EPSG:4326 (WGS 84) which is the standard CRS for H3.
gdf_11 = gpd.GeoDataFrame(removal_11_df, geometry='geometry', crs="EPSG:4326")

In [ ]:
# --- 6. Save the GeoDataFrame to a File ---
output_file = 'hex_native_land_validation_r11.geojson'

# GeoJSON is generally the most compatible and robust format for sharing.
# For a shapefile, you would use: gdf.to_file('h3_coverage_validation.shp')
try:
    gdf_11.to_file(output_file, driver='GeoJSON')
    print("-" * 50)
    print(f"Successfully created GeoJSON file: {output_file}")
    print(f"File saved with {len(gdf_11)} features.")
    print("You can now load this file into QGIS to visualize your H3 coverage areas.")
except Exception as e:
    print(f"Error saving file: {e}")

# Display a preview of the GeoDataFrame structure
print("\nGeoDataFrame Preview:")
print(gdf_11.head())

#### Native Lands

In [ ]:
data_fn = os.path.join(DATA_DIR, 'census\\tl_2024_us_aiannh\\tl_2024_us_aiannh.shp')

In [ ]:
nl_gdf = gpd.read_file(data_fn)

In [ ]:
nl_gdf.plot(figsize=(13,8))

In [ ]:
nl_gdf = nl_gdf.to_crs(3857)

In [ ]:
nl_gdf['geom_buff'] = nl_gdf.buffer(10000)

In [ ]:
nl_gdf['geometry'] = nl_gdf['geom_buff']

In [ ]:
del nl_gdf['geom_buff']

In [ ]:
nl_gdf = nl_gdf.to_crs(4326)

In [ ]:
nl_hex_ids_r5 = []

In [ ]:
try:
    tmp_hex_r5_df = nl_gdf.h3.polyfill(5, explode=True)
    nl_hex_ids_r5 +=  tmp_hex_r5_df.h3_polyfill.drop_duplicates().to_list()
    
except Exception as e:
    print(e)
    print("error with r5")

In [ ]:
h5_remove_list = [x for x in list(set(nl_hex_ids_r5)) if str(x) != 'nan']

In [ ]:
len(h5_remove_list)

In [ ]:
save_df = pd.DataFrame([{"hex_id": x, "reason": "native_land"} for x in h5_remove_list])

In [ ]:
save_df.to_csv('remove_hexes_r5_native_land.csv')

In [ ]:
# --- 3. Create a Pandas DataFrame ---
removal_df = pd.DataFrame({'hex_id': h5_remove_list})
print(f"Converting {len(removal_df)} H3 IDs to polygon geometries...")
removal_df['geometry'] = removal_df['hex_id'].apply(hex_polygon)

In [ ]:
# --- 5. Create the GeoDataFrame ---
# Use EPSG:4326 (WGS 84) which is the standard CRS for H3.
gdf = gpd.GeoDataFrame(removal_df, geometry='geometry', crs="EPSG:4326")

In [ ]:
# --- 6. Save the GeoDataFrame to a File ---
output_file = 'hex_native_land_validation_r5.geojson'

# GeoJSON is generally the most compatible and robust format for sharing.
# For a shapefile, you would use: gdf.to_file('h3_coverage_validation.shp')
try:
    gdf.to_file(output_file, driver='GeoJSON')
    print("-" * 50)
    print(f"Successfully created GeoJSON file: {output_file}")
    print(f"File saved with {len(gdf)} features.")
    print("You can now load this file into QGIS to visualize your H3 coverage areas.")
except Exception as e:
    print(f"Error saving file: {e}")

# Display a preview of the GeoDataFrame structure
print("\nGeoDataFrame Preview:")
print(gdf.head())